# Retraining YOLO26n v2 (832px, 80ep) on Google Colab T4 GPU
Target: Overall F1 ~0.90 with maximum boost on `no_helmet` class.


## 1. Install Ultralytics & Dependencies


In [ ]:
!pip install -q -U ultralytics roboflow
import ultralytics
ultralytics.checks()


## 2. Mount Google Drive


In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted at /content/drive')


## 3. Ingest Base Dataset into /content/datasets/merged


In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

dest = Path('/content/datasets/merged')
dest.mkdir(parents=True, exist_ok=True)

# Candidate paths in Google Drive
candidates = [
    Path('/content/drive/MyDrive/Colab Notebooks/traffictraining dataset/merged'),
    Path('/content/drive/MyDrive/traffictraining dataset/merged'),
    Path('/content/drive/MyDrive/merged.zip'),
    Path('/content/drive/MyDrive/datasets/merged.zip'),
]

found_base = None
for c in candidates:
    if c.exists():
        found_base = c
        break

if found_base is None:
    # Deep search in Drive
    for root, dirs, files in os.walk('/content/drive/MyDrive'):
        if 'traffictraining' in root.lower() and 'merged' in root.lower() and os.path.isdir(os.path.join(root, 'images')):
            found_base = Path(root)
            break
        for f in files:
            if 'merged' in f.lower() and f.endswith('.zip'):
                found_base = Path(root) / f
                break
        if found_base:
            break

if found_base is None:
    raise FileNotFoundError('Could not find base dataset in Google Drive! Check folder or upload merged.zip.')

print(f'Found base dataset at: {found_base}')
if found_base.is_dir():
    for sub in ['images', 'labels']:
        src_sub = found_base / sub
        dst_sub = dest / sub
        if not dst_sub.exists():
            print(f'Copying {src_sub} -> {dst_sub}...')
            shutil.copytree(src_sub, dst_sub)
else:
    print(f'Unzipping {found_base} -> {dest}...')
    with zipfile.ZipFile(found_base, 'r') as z:
        z.extractall(dest)

train_base_imgs = len(list((dest / 'images' / 'train').glob('*.*')))
val_base_imgs = len(list((dest / 'images' / 'val').glob('*.*')))
print(f'Base dataset ready: {train_base_imgs} train images, {val_base_imgs} val images (val kept untouched).')


## 4. Ingest Extra Helmet Datasets, Map Classes & Deduplicate


In [ ]:
import hashlib
import glob
import yaml
from pathlib import Path

extra_dir = Path('/content/drive/MyDrive/extra_helmet_datasets')
zips = list(extra_dir.glob('*.zip')) if extra_dir.exists() else []

print(f'Checking extra datasets folder: {extra_dir}')
print(f'Found {len(zips)} zip files.')

# Existing train MD5 hashes to prevent duplicates
existing_hashes = set()
for img_p in (dest / 'images' / 'train').glob('*.*'):
    with open(img_p, 'rb') as f:
        existing_hashes.add(hashlib.md5(f.read()).hexdigest())
for img_p in (dest / 'images' / 'val').glob('*.*'):
    with open(img_p, 'rb') as f:
        existing_hashes.add(hashlib.md5(f.read()).hexdigest())
print(f'Loaded {len(existing_hashes)} existing image hashes for deduplication.')

def map_class_name(name):
    n = name.lower().strip()
    if 'plate' in n or 'licen' in n:
        return 2
    if any(k in n for k in ['without', 'no helmet', 'nohelmet', 'not wearing', 'head']):
        return 1
    if 'helmet' in n:
        return 0
    return None

test_img_dir = dest / 'images' / 'test'
test_lbl_dir = dest / 'labels' / 'test'
test_img_dir.mkdir(parents=True, exist_ok=True)
test_lbl_dir.mkdir(parents=True, exist_ok=True)

# Process any provided zips in My Drive/extra_helmet_datasets/
for z_path in zips:
    unzip_tmp = Path('/content/extra_tmp') / z_path.stem
    unzip_tmp.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(z_path, 'r') as z:
        z.extractall(unzip_tmp)
    
    yaml_candidates = list(unzip_tmp.rglob('data*.yaml'))
    if not yaml_candidates:
        continue
    with open(yaml_candidates[0], 'r', encoding='utf-8') as yf:
        meta = yaml.safe_load(yf)
    names = meta.get('names', {})
    if isinstance(names, list):
        names = {i: n for i, n in enumerate(names)}
    
    mapping = {cid: map_class_name(cname) for cid, cname in names.items()}
    print(f'\n--- Class Mapping for {z_path.name} ---')
    for orig_id, orig_name in names.items():
        mapped_id = mapping.get(orig_id)
        target_name = {0: 'helmet', 1: 'no_helmet', 2: 'number_plate'}.get(mapped_id, 'DROPPED')
        print(f'  Orig [{orig_id}: {orig_name}] -> Target [{mapped_id}: {target_name}]')


## 5. Held-Out Test Split (300 Original Train Images -> Test)


In [ ]:
import random

train_imgs = sorted(list((dest / 'images' / 'train').glob('*.*')))
rng = random.Random(0)
sampled_test = set(rng.sample(train_imgs, 300))

print('Moving 300 original train images into test split (seed 0)...')
for img_path in sampled_test:
    stem = img_path.stem
    target_img = test_img_dir / img_path.name
    shutil.move(str(img_path), str(target_img))
    
    lbl_path = dest / 'labels' / 'train' / f'{stem}.txt'
    if lbl_path.exists():
        target_lbl = test_lbl_dir / f'{stem}.txt'
        shutil.move(str(lbl_path), str(target_lbl))

n_tr = len(list((dest / 'images' / 'train').glob('*.*')))
n_va = len(list((dest / 'images' / 'val').glob('*.*')))
n_te = len(list((dest / 'images' / 'test').glob('*.*')))
print(f'\nSplit summary: {n_tr} train | {n_va} val (UNTOUCHED) | {n_te} test')

# Rewrite data.yaml
yaml_content = f'''path: {dest}
train: images/train
val: images/val
test: images/test

nc: 3
names:
  0: helmet
  1: no_helmet
  2: number_plate
'''
yaml_file = dest / 'data.yaml'
yaml_file.write_text(yaml_content.strip() + '\n')
print(f'\nConfigured {yaml_file}:\n' + yaml_file.read_text())


## 6. Train YOLO26n v2 at 832px, 80 Epochs (T4 GPU)


In [ ]:
from ultralytics import YOLO

project_dir = Path('/content/drive/MyDrive/traffic_runs')
run_name = 'yolo26n_v2_832'
last_pt = project_dir / run_name / 'weights' / 'last.pt'

if last_pt.exists():
    print(f'Resuming training from: {last_pt}')
    model = YOLO(str(last_pt))
    results = model.train(resume=True)
else:
    print('Starting fresh training: yolo26n.pt @ 832px, 80 epochs, patience 20...')
    model = YOLO('yolo26n.pt')
    results = model.train(
        data=str(dest / 'data.yaml'),
        epochs=80,
        imgsz=832,
        batch=16,
        patience=20,
        seed=0,
        device=0,
        project=str(project_dir),
        name=run_name,
        exist_ok=True,
    )


## 7. Evaluate on Both Val & Test Splits & Export to Drive


In [ ]:
import json

best_pt = project_dir / run_name / 'weights' / 'best.pt'
if not best_pt.exists():
    best_pt = project_dir / run_name / 'weights' / 'last.pt'

eval_model = YOLO(str(best_pt))

# 1. Validation split evaluation (comparable with 0.877 baseline)
print('Evaluating on VAL split @ 832px...')
val_res = eval_model.val(data=str(dest / 'data.yaml'), split='val', imgsz=832, device=0)
v_p = round(float(val_res.box.p.mean()), 4)
v_r = round(float(val_res.box.r.mean()), 4)
v_f1 = round(float(2 * v_p * v_r / max(1e-6, v_p + v_r)), 4)
v_map50 = round(float(val_res.box.map50), 4)
v_map = round(float(val_res.box.map), 4)

classes = ['helmet', 'no_helmet', 'number_plate']
val_per_class = {}
for i, cname in enumerate(classes):
    cp = round(float(val_res.box.p[i]), 4) if i < len(val_res.box.p) else 0.0
    cr = round(float(val_res.box.r[i]), 4) if i < len(val_res.box.r) else 0.0
    cf1 = round(float(2 * cp * cr / max(1e-6, cp + cr)), 4)
    cmap50 = round(float(val_res.box.maps[i]), 4) if i < len(val_res.box.maps) else 0.0
    val_per_class[cname] = {'precision': cp, 'recall': cr, 'f1': cf1, 'mAP50': cmap50}

# 2. Test split evaluation (honest unseen test score)
print('Evaluating on TEST split @ 832px...')
test_res = eval_model.val(data=str(dest / 'data.yaml'), split='test', imgsz=832, device=0)
t_p = round(float(test_res.box.p.mean()), 4)
t_r = round(float(test_res.box.r.mean()), 4)
t_f1 = round(float(2 * t_p * t_r / max(1e-6, t_p + t_r)), 4)
t_map50 = round(float(test_res.box.map50), 4)
t_map = round(float(test_res.box.map), 4)

test_per_class = {}
for i, cname in enumerate(classes):
    cp = round(float(test_res.box.p[i]), 4) if i < len(test_res.box.p) else 0.0
    cr = round(float(test_res.box.r[i]), 4) if i < len(test_res.box.r) else 0.0
    cf1 = round(float(2 * cp * cr / max(1e-6, cp + cr)), 4)
    cmap50 = round(float(test_res.box.maps[i]), 4) if i < len(test_res.box.maps) else 0.0
    test_per_class[cname] = {'precision': cp, 'recall': cr, 'f1': cf1, 'mAP50': cmap50}

speed = val_res.speed

export_data = {
    'model': 'yolo26n_v2_832.pt',
    'overall': {
        'precision': v_p,
        'recall': v_r,
        'f1': v_f1,
        'mAP50': v_map50,
        'mAP50_95': v_map,
    },
    'speed_ms_per_image': speed,
    'per_class': val_per_class,
    'test': {
        'precision': t_p,
        'recall': t_r,
        'f1': t_f1,
        'mAP50': t_map50,
        'mAP50_95': t_map,
        'per_class': test_per_class,
    }
}

out_dir = Path('/content/drive/MyDrive/traffic_model_output/yolo26n_v2_832')
out_dir.mkdir(parents=True, exist_ok=True)

# Copy artifacts to Drive
shutil.copy2(best_pt, out_dir / 'best.pt')
for f in ['results.csv', 'results.png', 'confusion_matrix_normalized.png']:
    fp = project_dir / run_name / f
    if fp.exists():
        shutil.copy2(fp, out_dir / f)

json_out = out_dir / 'evaluation_results_yolo26n_v2_832.json'
with open(json_out, 'w', encoding='utf-8') as f:
    json.dump(export_data, f, indent=2)

print(f'\n✅ All results and weights exported to {out_dir}:')
print(json.dumps(export_data, indent=2))
